# scRNA-seq: fibroblast (CAF) subtypes (Fig. 2D, 2E, 2I; Supplementary Fig. S6D, S6G)
- **Fig. 2D** UMAP of CAF subpopulations
- **Fig. 2E** CAF subpopulation proportions per group
- **Fig. 2I** Acta2 and Lrrc15 expression in fibroblasts
- **Supplementary Fig. S6D** CAF marker dot plot
- **Supplementary Fig. S6G** signatures increased in fibroblasts from the Running group

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

from sc_utils import toCount_HVG_Scale

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3
sc.logging.print_header()
sc.set_figure_params(dpi = 100, dpi_save = 300, transparent=True, color_map = 'OrRd')

DATA_DIR = '../data'   # Melanoma_scRNAseq_data.h5ad (GEO: GSE289587 / Zenodo)

In [ ]:
adata = sc.read(f"{DATA_DIR}/Melanoma_scRNAseq_data.h5ad")
adata

## Subset and re-embed fibroblasts (Harmony by sample)

In [ ]:
Fibro = adata[adata.obs['Level2_annotation'].str.startswith('FB')].copy()

In [ ]:
Fibro = toCount_HVG_Scale(Fibro)

In [ ]:
sc.tl.pca(Fibro)
sc.pl.pca_variance_ratio(Fibro, log=True)

In [ ]:
sc.pp.neighbors(Fibro, n_pcs=20)
sc.tl.umap(Fibro)

In [ ]:
sc.external.pp.harmony_integrate(Fibro, key = 'sample', max_iter_harmony = 30)

In [ ]:
sc.pp.neighbors(Fibro, n_pcs=20, use_rep = 'X_pca_harmony')
sc.tl.umap(Fibro)

## Fig. 2D — UMAP

In [ ]:
sc.pl.umap(Fibro, color = 'Level2_annotation', frameon = False, title = '')

## Fig. 2E — Proportions

In [ ]:
proportions = Fibro.obs.groupby(['sample', 'Level2_annotation']).size().unstack(fill_value=0)
proportions = proportions.div(proportions.sum(axis=1), axis=0)
proportions = proportions * 100

In [ ]:
cell_type_order = 'FB_Lrrc15+,FB_Col15a1+,FB_Pi16+,FB_Mki67+,FB_Vegfa+,FB_Isg15+'.split(',')
proportions = proportions[cell_type_order]

In [ ]:
proportions_with_group = proportions.reset_index()
proportions_with_group['group'] = proportions_with_group['sample'].map({
    'S1': 'Mattress', 'S2': 'Mattress', 'Sponge_1': 'Mattress', 'Sponge_3': 'Mattress',
    'T1': 'Running', 'T2': 'Running', 'Tread_1': 'Running', 'Tread_2': 'Running', 'Tread_3': 'Running'
})

In [ ]:
melted_data = pd.melt(proportions_with_group, id_vars=['sample', 'group'], var_name='cell_type', value_name='proportion')
melted_data['cell_type'] = pd.Categorical(melted_data['cell_type'], categories=cell_type_order, ordered=True)

In [ ]:
plt.figure(figsize=(20, 10))

g = sns.barplot(
    data=melted_data,
    x="cell_type",
    y="proportion",
    hue="group",
    palette=['#D4D4D4CC', '#0000FFCC'],
    width=0.8
)

g.grid(False)

g.spines['left'].set_color('black')
g.spines['bottom'].set_color('black')
g.spines['left'].set_linewidth(2)
g.spines['bottom'].set_linewidth(2)

g.spines['top'].set_visible(False)
g.spines['right'].set_visible(False)

plt.xticks(rotation=60, fontsize=20)

plt.yticks(fontsize=20)

plt.ylabel("Cell type proportion (%)", fontsize=24)
plt.xlabel("", fontsize=24)

plt.legend(title="Group", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=20)
plt.setp(g.get_legend().get_title(), fontsize=20)

plt.tight_layout()
plt.show()

## Supplementary Fig. S6D — Marker genes

In [ ]:
sc.pl.dotplot(Fibro, ['Col15a1', 'Isg15', 'Lrrc15', 'Mki67', 'Pi16', 'Vegfa'], groupby='Level2_annotation', standard_scale='var')

## Fig. 2I — Acta2 and Lrrc15 expression in fibroblasts
Log-normalized expression (`adata.raw`) in all fibroblasts (`Level1_annotation == 'Fibroblast'`).
The manuscript panels were drawn with ggplot2 (`geom_violin(draw_quantiles = 0.5)`); `plot_group_violin` reproduces them
(density, violin width and the median line computed as in ggplot2).

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu
from sc_utils import plot_group_violin

fb = adata[adata.obs['Level1_annotation'] == 'Fibroblast']
g = fb.obs['group'].map({'Sponge': 'Mattress', 'Tread': 'Running'}).astype(str).values
for gene in ['Acta2', 'Lrrc15']:
    x = np.asarray(fb.raw[:, gene].X.todense()).ravel()
    a, b = x[g == 'Mattress'], x[g == 'Running']
    print(f'{gene}: Mattress n = {len(a)}, Running n = {len(b)}; '
          f't-test p = {ttest_ind(a, b)[1]:.3g}, Mann-Whitney U p = {mannwhitneyu(a, b)[1]:.3g}')
    plot_group_violin(x, g, gene, ylabel='Mean expression')
    plt.show()

## Supplementary Fig. S6G — Signatures increased in fibroblasts from the Running group
AUCell scores (decoupler, `min_n=1`) of fibroblasts (Level2_annotation FB_*), averaged per sample; Running (n = 5) vs Mattress (n = 4),
two-sided Student's t-test of the sample means. Shown are the signatures selected for the figure
(log2FC > 0 and P < 0.05).

In [ ]:
import json
from sc_utils import plot_hbar, signature_group_test

with open(f'{DATA_DIR}/genesets_used.json', 'r') as f:
    genesets = json.load(f)
S6G_TERMS = {'REACTOME_CELL_EXTRACELLULAR_MATRIX_INTERACTIONS': 'Cell-extracellular matrix interactions',
             'WP_ALPHA_6_BETA_4_INTEGRIN_SIGNALING_PATHWAY': 'Alpha 6 beta 4 integrin signaling pathway',
             'GOBP_ACTIN_MEDIATED_CELL_CONTRACTION': 'Actin mediated cell contraction',
             'GOBP_INSULIN_RECEPTOR_SIGNALING_PATHWAY': 'Insulin receptor signaling pathway',
             'REACTOME_SIGNALING_BY_RECEPTOR_TYROSINE_KINASES': 'Signaling by Receptor Tyrosine Kinase'}

Fibro_sig = adata[adata.obs['Level2_annotation'].str.startswith('FB')].copy()
del Fibro_sig.X
Fibro_sig = toCount_HVG_Scale(Fibro_sig)
res = signature_group_test(Fibro_sig, genesets, list(S6G_TERMS)).sort_values('-log10P', ascending=False)
display(res)
fig, ax = plot_hbar(res['-log10P'].values, [S6G_TERMS[k] for k in res.index], color='#3333FF',
                    xlabel='-log(P-value)', figsize=(3, 2))
fig.set_label('Fibroblast_signatures')
plt.show()